In [ ]:
from nse_platform.context.execution_context import ExecutionContext
from pyspark.sql import functions as F


ec = ExecutionContext.for_local_spark("explore", "2023-05-24")
spark = ec.spark


# NEW UDIFF xplore

In [53]:
df = spark.read.parquet("../data/bronze/equity/nse/bhavcopy/year=2026/month=01/day=01")

In [49]:

df.printSchema()                       


root
 |-- TradDt: string (nullable = true)
 |-- BizDt: string (nullable = true)
 |-- Sgmt: string (nullable = true)
 |-- Src: string (nullable = true)
 |-- FinInstrmTp: string (nullable = true)
 |-- FinInstrmId: string (nullable = true)
 |-- ISIN: string (nullable = true)
 |-- TckrSymb: string (nullable = true)
 |-- SctySrs: string (nullable = true)
 |-- XpryDt: string (nullable = true)
 |-- FininstrmActlXpryDt: string (nullable = true)
 |-- StrkPric: string (nullable = true)
 |-- OptnTp: string (nullable = true)
 |-- FinInstrmNm: string (nullable = true)
 |-- OpnPric: string (nullable = true)
 |-- HghPric: string (nullable = true)
 |-- LwPric: string (nullable = true)
 |-- ClsPric: string (nullable = true)
 |-- LastPric: string (nullable = true)
 |-- PrvsClsgPric: string (nullable = true)
 |-- UndrlygPric: string (nullable = true)
 |-- SttlmPric: string (nullable = true)
 |-- OpnIntrst: string (nullable = true)
 |-- ChngInOpnIntrst: string (nullable = true)
 |-- TtlTradgVol: string (n

In [50]:
df.show(10, truncate=False)              


+----------+----------+----+---+-----------+-----------+------------+----------+-------+------+-------------------+--------+------+------------------------+--------+--------+--------+--------+--------+------------+-----------+---------+---------+---------------+-----------+----------+---------------+-----+------------+----+-----+-----+-----+-----+
|TradDt    |BizDt     |Sgmt|Src|FinInstrmTp|FinInstrmId|ISIN        |TckrSymb  |SctySrs|XpryDt|FininstrmActlXpryDt|StrkPric|OptnTp|FinInstrmNm             |OpnPric |HghPric |LwPric  |ClsPric |LastPric|PrvsClsgPric|UndrlygPric|SttlmPric|OpnIntrst|ChngInOpnIntrst|TtlTradgVol|TtlTrfVal |TtlNbOfTxsExctd|SsnId|NewBrdLotQty|Rmks|Rsvd1|Rsvd2|Rsvd3|Rsvd4|
+----------+----------+----+---+-----------+-----------+------------+----------+-------+------+-------------------+--------+------+------------------------+--------+--------+--------+--------+--------+------------+-----------+---------+---------+---------------+-----------+----------+---------------

In [51]:
print("total rows:", df.count())
df.groupBy("SctySrs").agg(F.count("*").alias("count")).orderBy(F.col("count").desc()).show() 

total rows: 3227
+-------+-----+
|SctySrs|count|
+-------+-----+
|     EQ| 2383|
|     SM|  361|
|     BE|  111|
|     ST|   73|
|     GS|   50|
|     GB|   44|
|     BZ|   33|
|     N0|   18|
|     TB|   10|
|     N2|   10|
|     NA|    9|
|     IV|    9|
|     N1|    9|
|     N5|    8|
|     E1|    8|
|     N8|    7|
|     N6|    7|
|     N9|    5|
|     RR|    5|
|     N7|    5|
+-------+-----+
only showing top 20 rows


In [56]:
df = spark.read.parquet("../data/bronze/equity/nse/bhavcopy/year=2026/month=01/day=01")
df= df.filter(F.col("SctySrs").isin("BE","EQ")).na.drop()
df.show()

+------+-----+----+---+-----------+-----------+----+--------+-------+------+-------------------+--------+------+-----------+-------+-------+------+-------+--------+------------+-----------+---------+---------+---------------+-----------+---------+---------------+-----+------------+----+-----+-----+-----+-----+
|TradDt|BizDt|Sgmt|Src|FinInstrmTp|FinInstrmId|ISIN|TckrSymb|SctySrs|XpryDt|FininstrmActlXpryDt|StrkPric|OptnTp|FinInstrmNm|OpnPric|HghPric|LwPric|ClsPric|LastPric|PrvsClsgPric|UndrlygPric|SttlmPric|OpnIntrst|ChngInOpnIntrst|TtlTradgVol|TtlTrfVal|TtlNbOfTxsExctd|SsnId|NewBrdLotQty|Rmks|Rsvd1|Rsvd2|Rsvd3|Rsvd4|
+------+-----+----+---+-----------+-----------+----+--------+-------+------+-------------------+--------+------+-----------+-------+-------+------+-------+--------+------------+-----------+---------+---------+---------------+-----------+---------+---------------+-----+------------+----+-----+-----+-----+-----+
+------+-----+----+---+-----------+-----------+----+--------+---

In [38]:
COLUMN_MAP_NEW = {
            "TckrSymb": "symbol",
            "SctySrs": "series",
            "TradDt": "trade_date",
            "OpnPric": "open",
            "HghPric": "high",
            "LwPric": "low",
            "ClsPric": "close",
            "LastPric": "last",
            "PrvsClsgPric": "prev_close",
            "TtlTradgVol": "total_traded_qty",
            "TtlTrfVal": "total_traded_value",
            "TtlNbOfTxsExctd": "total_trades",
            "ISIN": "isin",
        }

df = df.withColumnsRenamed(COLUMN_MAP_NEW).select([v for k,v in COLUMN_MAP_NEW.items()])
df.show(10, truncate=False)   

+----------+------+----------+--------+--------+--------+--------+--------+----------+----------------+------------------+------------+------------+
|symbol    |series|trade_date|open    |high    |low     |close   |last    |prev_close|total_traded_qty|total_traded_value|total_trades|isin        |
+----------+------+----------+--------+--------+--------+--------+--------+----------+----------------+------------------+------------+------------+
|SGBJUN28  |GB    |2026-01-01|14311.00|14444.99|14247.62|14396.15|14424.00|14305.84  |587             |8434306.46        |96          |IN0020200104|
|SGBN28VIII|GB    |2026-01-01|14468.00|14514.63|14403.01|14514.03|14514.63|14469.60  |135             |1949525.34        |51          |IN0020200286|
|SGBMAY29I |GB    |2026-01-01|14500.00|14619.90|14466.00|14534.41|14587.00|14502.61  |576             |8364148.59        |161         |IN0020210053|
|SGBJUN29II|GB    |2026-01-01|14563.76|14798.00|14476.19|14506.59|14525.00|14491.31  |393             |572

In [39]:
PRICE_COLS = ["open", "high", "low", "close", "last", "prev_close", "total_traded_value"]
LONG_COLS = ["total_traded_qty", "total_trades"]

df = df.withColumns({c: F.col(c).cast("double") for c in PRICE_COLS})
df = df.withColumns({c: F.col(c).cast("long") for c in LONG_COLS})

In [40]:
df.select("trade_date").show(5, truncate=False)

+----------+
|trade_date|
+----------+
|2026-01-01|
|2026-01-01|
|2026-01-01|
|2026-01-01|
|2026-01-01|
+----------+
only showing top 5 rows


In [41]:
df= df.withColumn("trade_date", F.col("trade_date").cast("date"))

In [42]:
df.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_new_udiff.columns]).show()

+------+------+----------+----+----+---+-----+----+----------+----------------+------------------+------------+----+
|symbol|series|trade_date|open|high|low|close|last|prev_close|total_traded_qty|total_traded_value|total_trades|isin|
+------+------+----------+----+----+---+-----+----+----------+----------------+------------------+------------+----+
|     0|     0|         0|   0|   0|  0|    0|   0|         0|               0|                 0|           0|   0|
+------+------+----------+----+----+---+-----+----+----------+----------------+------------------+------------+----+



In [43]:
print(df.count())
print(df.na.drop().count())

3227
3227


In [44]:
df.groupBy("series").count().orderBy(F.desc("count")).show(20, truncate=False)

+------+-----+
|series|count|
+------+-----+
|EQ    |2383 |
|SM    |361  |
|BE    |111  |
|ST    |73   |
|GS    |50   |
|GB    |44   |
|BZ    |33   |
|N0    |18   |
|TB    |10   |
|N2    |10   |
|NA    |9    |
|IV    |9    |
|N1    |9    |
|N5    |8    |
|E1    |8    |
|N8    |7    |
|N6    |7    |
|N9    |5    |
|RR    |5    |
|N7    |5    |
+------+-----+
only showing top 20 rows


In [45]:
df.groupBy("symbol", "trade_date").count().filter("count > 1").show()

+---------+----------+-----+
|   symbol|trade_date|count|
+---------+----------+-----+
|RADIOCITY|2026-01-01|    2|
+---------+----------+-----+



In [46]:
df.filter((F.col("symbol") == "RADIOCITY") & (F.col("trade_date") == "2026-01-01")).show(truncate=False)

+---------+------+----------+-----+-----+-----+-----+-----+----------+----------------+------------------+------------+------------+
|symbol   |series|trade_date|open |high |low  |close|last |prev_close|total_traded_qty|total_traded_value|total_trades|isin        |
+---------+------+----------+-----+-----+-----+-----+-----+----------+----------------+------------------+------------+------------+
|RADIOCITY|EQ    |2026-01-01|6.88 |6.9  |6.63 |6.75 |6.73 |6.72      |68310           |459644.4          |311         |INE919I01024|
|RADIOCITY|P1    |2026-01-01|119.2|119.2|119.2|119.2|119.2|118.05    |50              |5960.0            |1           |INE919I04010|
+---------+------+----------+-----+-----+-----+-----+-----+----------+----------------+------------------+------------+------------+



# FIX

## fix new uidff

In [32]:

from pyspark.sql import Window

df_new_udiff = spark.read.parquet("../data/bronze/equity/nse/bhavcopy/year=2026/month=01/day=01")
df_new_udiff.filter(F.col("SctySrs").isin("BE","EQ")).na.drop()

df = df_new_udiff.withColumnsRenamed(COLUMN_MAP_NEW).select([v for k,v in COLUMN_MAP_NEW.items()])

PRICE_COLS = ["open", "high", "low", "close", "last", "prev_close", "total_traded_value"]
LONG_COLS = ["total_traded_qty", "total_trades"]
df = df.withColumns({c: F.col(c).cast("double") for c in PRICE_COLS})
df = df.withColumns({c: F.col(c).cast("long") for c in LONG_COLS})
df= df.withColumn("trade_date", F.col("trade_date").cast("date"))


w = Window.partitionBy("symbol", "trade_date").orderBy(F.col("trade_date"))
df = df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")
df.show(10, truncate= False)


+----------+------+----------+--------+--------+--------+--------+--------+----------+----------------+------------------+------------+------------+
|symbol    |series|trade_date|open    |high    |low     |close   |last    |prev_close|total_traded_qty|total_traded_value|total_trades|isin        |
+----------+------+----------+--------+--------+--------+--------+--------+----------+----------------+------------------+------------+------------+
|0SCL26    |YR    |2026-01-01|1163.6  |1163.6  |1163.0  |1163.0  |1163.0  |1135.2    |20              |23266.0           |2           |INE148I07SF0|
|1003SCFL31|Z4    |2026-01-01|979.9   |979.9   |979.9   |979.9   |979.9   |984.0     |30              |29397.0           |2           |INE148I07SX3|
|1015ECL29 |NR    |2026-01-01|1024.0  |1024.0  |1022.0  |1022.0  |1022.0  |1016.7    |9               |9200.0            |3           |INE804IA7014|
|1015UCL26 |N1    |2026-01-01|996.05  |996.05  |996.05  |996.05  |996.05  |996.0     |10              |996

# explore old udiff

In [23]:
df_old_udiff = spark.read.parquet("../data/bronze/equity/nse/bhavcopy/year=2023/month=05/day=24")
df_old_udiff.printSchema()

root
 |-- SYMBOL: string (nullable = true)
 |-- SERIES: string (nullable = true)
 |-- OPEN: string (nullable = true)
 |-- HIGH: string (nullable = true)
 |-- LOW: string (nullable = true)
 |-- CLOSE: string (nullable = true)
 |-- LAST: string (nullable = true)
 |-- PREVCLOSE: string (nullable = true)
 |-- TOTTRDQTY: string (nullable = true)
 |-- TOTTRDVAL: string (nullable = true)
 |-- TIMESTAMP: string (nullable = true)
 |-- TOTALTRADES: string (nullable = true)
 |-- ISIN: string (nullable = true)
 |-- _c13: string (nullable = true)



In [24]:
df_old_udiff.show(10,truncate=False)

+----------+------+-----+-----+-----+-----+-----+---------+---------+-----------+-----------+-----------+------------+----+
|SYMBOL    |SERIES|OPEN |HIGH |LOW  |CLOSE|LAST |PREVCLOSE|TOTTRDQTY|TOTTRDVAL  |TIMESTAMP  |TOTALTRADES|ISIN        |_c13|
+----------+------+-----+-----+-----+-----+-----+---------+---------+-----------+-----------+-----------+------------+----+
|1018GS2026|GS    |113  |113  |113  |113  |113  |114      |1        |113        |24-MAY-2023|1          |IN0020010081|NULL|
|182D220923|TB    |97.6 |97.6 |97.6 |97.6 |97.6 |97.6     |500      |48800      |24-MAY-2023|1          |IN002022Y526|NULL|
|182D240823|TB    |97.86|97.86|97.86|97.86|97.86|97.86    |500      |48930      |24-MAY-2023|1          |IN002022Y484|NULL|
|20MICRONS |EQ    |87.1 |95   |86.55|91.85|92.4 |87.75    |910681   |84035918.25|24-MAY-2023|10750      |INE144J01027|NULL|
|21STCENMGM|EQ    |18.55|18.9 |18.55|18.6 |18.6 |18.55    |1839     |34558.8    |24-MAY-2023|18         |INE253B01015|NULL|
|360ONE 

In [25]:
df_old_udiff.filter(F.col("SERIES").isin("BE","EQ")).na.drop().show()

+------+------+----+----+---+-----+----+---------+---------+---------+---------+-----------+----+----+
|SYMBOL|SERIES|OPEN|HIGH|LOW|CLOSE|LAST|PREVCLOSE|TOTTRDQTY|TOTTRDVAL|TIMESTAMP|TOTALTRADES|ISIN|_c13|
+------+------+----+----+---+-----+----+---------+---------+---------+---------+-----------+----+----+
+------+------+----+----+---+-----+----+---------+---------+---------+---------+-----------+----+----+



In [26]:
COLUMN_MAP_OLD = {
            "SYMBOL": "symbol",
            "SERIES": "series",
            "TIMESTAMP": "trade_date",
            "OPEN": "open",
            "HIGH": "high",
            "LOW": "low",
            "CLOSE": "close",
            "LAST": "last",
            "PREVCLOSE": "prev_close",
            "TOTTRDQTY": "total_traded_qty",
            "TOTTRDVAL": "total_traded_value",
            "TOTALTRADES": "total_trades",
            "ISIN": "isin",
        }
df_old_udiff = df_old_udiff.withColumnsRenamed(COLUMN_MAP_OLD).select([v for k,v in COLUMN_MAP_OLD.items()])
df_old_udiff.show(10, truncate=False) 

+----------+------+-----------+-----+-----+-----+-----+-----+----------+----------------+------------------+------------+------------+
|symbol    |series|trade_date |open |high |low  |close|last |prev_close|total_traded_qty|total_traded_value|total_trades|isin        |
+----------+------+-----------+-----+-----+-----+-----+-----+----------+----------------+------------------+------------+------------+
|1018GS2026|GS    |24-MAY-2023|113  |113  |113  |113  |113  |114       |1               |113               |1           |IN0020010081|
|182D220923|TB    |24-MAY-2023|97.6 |97.6 |97.6 |97.6 |97.6 |97.6      |500             |48800             |1           |IN002022Y526|
|182D240823|TB    |24-MAY-2023|97.86|97.86|97.86|97.86|97.86|97.86     |500             |48930             |1           |IN002022Y484|
|20MICRONS |EQ    |24-MAY-2023|87.1 |95   |86.55|91.85|92.4 |87.75     |910681          |84035918.25       |10750       |INE144J01027|
|21STCENMGM|EQ    |24-MAY-2023|18.55|18.9 |18.55|18.6 |

In [27]:
print(df_old_udiff.count())
print(df_old_udiff.na.drop().count())

2377
2377


In [28]:

PRICE_COLS = ["open", "high", "low", "close", "last", "prev_close", "total_traded_value"]
LONG_COLS = ["total_traded_qty", "total_trades"]
df_old_udiff = df_old_udiff.withColumns({c: F.col(c).cast("double") for c in PRICE_COLS})
df_old_udiff = df_old_udiff.withColumns({c: F.col(c).cast("long") for c in LONG_COLS})
df_old_udiff.select("trade_date").show(5, truncate=False)




+-----------+
|trade_date |
+-----------+
|24-MAY-2023|
|24-MAY-2023|
|24-MAY-2023|
|24-MAY-2023|
|24-MAY-2023|
+-----------+
only showing top 5 rows


In [29]:
df_old_udiff= df_old_udiff.withColumn("trade_date", F.to_date("trade_date", "dd-MMM-yyyy"))

In [30]:
df_old_udiff.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_old_udiff.columns]).show()

+------+------+----------+----+----+---+-----+----+----------+----------------+------------------+------------+----+
|symbol|series|trade_date|open|high|low|close|last|prev_close|total_traded_qty|total_traded_value|total_trades|isin|
+------+------+----------+----+----+---+-----+----+----------+----------------+------------------+------------+----+
|     0|     0|         0|   0|   0|  0|    0|   0|         0|               0|                 0|           0|   0|
+------+------+----------+----+----+---+-----+----+----------+----------------+------------------+------------+----+



## fix old udiff

In [31]:
df_old_udiff.filter(F.col("SERIES").isin("BE","EQ")).na.drop()
df_old_udiff = df_old_udiff.withColumnsRenamed(COLUMN_MAP_OLD).select([v for k,v in COLUMN_MAP_OLD.items()])
df_old_udiff= df_old_udiff.withColumn("trade_date", F.to_date("trade_date", "dd-MMM-yyyy"))